In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
import torch

c:\PycharmProjects\projects\fine_tune_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1、构建一个BitsAndBytesConfig对象
bits_and_bytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # 使用nf4来进行量化
    bnb_4bit_quant_type="nf4",
    # 使用双重量化
    bnb_4bit_use_double_quant=True,
    # 计算时的数据类型
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [3]:
# 2、加载模型时，使用bits_and_bytes_config
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B",quantization_config = bits_and_bytes_config).to("cuda")

W0703 16:33:06.004000 26940 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 311/311 [00:01<00:00, 307.90it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [4]:
torch.cuda.memory_allocated() / 1024 ** 3

0.7978248596191406

In [5]:
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

In [13]:
from datasets import load_dataset
train_data = load_dataset("json",data_files={"train":"data/psychology_data.jsonl"})
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B")

In [14]:
# 2、处理数据集的格式
from typing  import List,Dict
def convert_data_format(examples:Dict[str, List]):
    """
    将数据集处理出 {"messages":[xx]}
    """
    coversation:List[List] = examples["conversation"]
    all_examples_messages:List = []

    for example in coversation:
        message_list = []
        example_message:Dict[str,str]=example[0]
        message_list.append({"role":"user","content":example_message["human"]})
        message_list.append({"role":"assistant","content":example_message["assistant"]})
        all_examples_messages.append(message_list)

    return {"messages":all_examples_messages}

mapped_train_data = train_data.map(convert_data_format,batched=True,remove_columns=['conversation_id', 'category', 'conversation', 'dataset'])


# 扩展：查看数据集的长度分布情况
def get_data_length(examples:Dict[str,list]):
    """
    获取到数据，经过apply chat template之后的长度
    """
    messages_lists = examples["messages"]
    length_list = []
    for message_list in messages_lists:
        result = tokenizer.apply_chat_template(message_list,tokenize=True)
        length_list.append(len(result["input_ids"]))
    return {"length":length_list}

data_with_length = mapped_train_data.map(get_data_length,batched=True,remove_columns=["messages"])

Map: 100%|██████████| 77250/77250 [00:25<00:00, 3048.61 examples/s]


In [17]:
pandas_length = data_with_length["train"].to_pandas()

In [20]:
pandas_length["length"].quantile(0.999)

np.float64(356.75100000000384)